# LeetCode #113 — Path Sum II: liệt kê mọi đường gốc → lá có tổng bằng target

**Mức độ:** Medium · **Chủ đề:** Backtracking, Tree, Depth-First Search, Binary Tree

Skeleton trên LeetCode:

```python
class Solution:
    def pathSum(self, root: Optional[TreeNode], targetSum: int) -> List[List[int]]:
```

Mục tiêu của notebook:

1. DFS từ gốc, mang theo **đường đi hiện tại** (`path`) và **tổng còn lại** (`remain`).
2. Kỹ thuật backtracking trên cây: `append` khi vào nút, `pop` khi ra.
3. Điều kiện **lá** đúng (`not left and not right`) — và vì sao không được dừng khi tổng vừa đủ.
4. Phiên bản lặp bằng stack, và liên hệ với #112, #437.

# 1. Bài toán là gì?

Cho `root` của cây nhị phân và số `targetSum`. Trả về **mọi** đường đi **từ gốc tới lá** sao cho tổng các giá trị trên đường bằng `targetSum`. Mỗi đường là một list **giá trị** (không phải tham chiếu nút).

**Lá** là nút **không có con**.

**Constraints (theo đề):**

- Số nút trong khoảng `[0, 5000]`.
- `-1000 <= Node.val <= 1000`
- `-1000 <= targetSum <= 1000`

---

## Ví dụ

```text
            5
           / \
          4   8
         /   / \
        11  13  4
       /  \    / \
      7    2  5   1

targetSum = 22  ->  [[5, 4, 11, 2], [5, 8, 4, 5]]

root = [1, 2, 3], targetSum = 5  ->  []
root = [1, 2],    targetSum = 0  ->  []
```

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Cây nhị phân (binary tree)
2. Duyệt theo chiều sâu (DFS — Depth-First Search)
3. Quay lui (backtracking)

Cuối mục có phần **2.4 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Cây nhị phân (binary tree)

### Cây nhị phân là gì?

Cây nhị phân gồm các **nút**; mỗi nút có một giá trị và tối đa **hai con**: con **trái** và con **phải**. Nút trên cùng là **gốc** (root); nút không có con là **lá** (leaf).

```text
          1          ← gốc (độ sâu 0)
        /   \
       2     3       ← độ sâu 1
      / \     \
     4   5     6     ← lá 4, 5, 6 (độ sâu 2)
```

Thuật ngữ: **cha/con**, **anh em** (cùng cha), **cây con** (một nút cùng toàn bộ con cháu), **độ sâu** (số cạnh từ gốc), **chiều cao** (số tầng / đường dài nhất từ gốc tới lá).

In [1]:
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

# Dựng cây ở hình trên bằng tay
goc = TreeNode(1,
               TreeNode(2, TreeNode(4), TreeNode(5)),
               TreeNode(3, None, TreeNode(6)))
print("gốc:", goc.val, "| con trái:", goc.left.val, "| con phải:", goc.right.val,
      "| 3 có con trái?", goc.right.left)

gốc: 1 | con trái: 2 | con phải: 3 | 3 có con trái? None


### Biểu diễn kiểu LeetCode: danh sách theo tầng (level-order)

LeetCode ghi cây thành danh sách đọc từng tầng, từ trái sang phải, `None` cho chỗ trống: cây trên là `[1, 2, 3, 4, 5, None, 6]`. Dựng cây từ danh sách này bằng một **hàng đợi**: lần lượt lấy nút cha ra, gán hai giá trị kế tiếp làm con trái/phải.

In [2]:
from collections import deque

def tao_cay(values):
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    q, i = deque([root]), 1
    while q and i < len(values):
        node = q.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i]); q.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i]); q.append(node.right)
        i += 1
    return root

def ve_cay(node, pre="", la_phai=True, goc_cay=True):
    """In cây nằm ngang để dễ nhìn."""
    if node is None:
        return
    ve_cay(node.right, pre + ("" if goc_cay else ("    " if la_phai else "│   ")), True, False)
    print(pre + ("" if goc_cay else ("┌── " if la_phai else "└── ")) + str(node.val))
    ve_cay(node.left, pre + ("" if goc_cay else ("│   " if la_phai else "    ")), False, False)

ve_cay(tao_cay([1, 2, 3, 4, 5, None, 6]))

    ┌── 6
┌── 3
1
│   ┌── 5
└── 2
    └── 4


(Cây in nằm ngang: gốc ở bên trái, con **phải** ở phía **trên**, con **trái** ở phía **dưới**.)

### Ba cách duyệt theo chiều sâu

Với mỗi nút ta làm ba việc: thăm **nút**, duyệt cây con **trái**, duyệt cây con **phải**. Thứ tự của ba việc cho ba kiểu duyệt:

| Kiểu | Thứ tự | Cây trên |
|---|---|---|
| **Preorder** (tiền thứ tự) | nút → trái → phải | `1 2 4 5 3 6` |
| **Inorder** (trung thứ tự) | trái → nút → phải | `4 2 5 1 3 6` |
| **Postorder** (hậu thứ tự) | trái → phải → nút | `4 5 2 6 3 1` |

Mẹo nhớ: "pre/in/post" chỉ **vị trí của nút** so với hai cây con.

In [3]:
def preorder(node, out):
    if node is None:
        return
    out.append(node.val)        # nút
    preorder(node.left, out)    # trái
    preorder(node.right, out)   # phải

def inorder(node, out):
    if node is None:
        return
    inorder(node.left, out)
    out.append(node.val)
    inorder(node.right, out)

def postorder(node, out):
    if node is None:
        return
    postorder(node.left, out)
    postorder(node.right, out)
    out.append(node.val)

cay = tao_cay([1, 2, 3, 4, 5, None, 6])
for ten, f in [("preorder", preorder), ("inorder", inorder), ("postorder", postorder)]:
    out = []; f(cay, out); print(f"{ten:<9}:", out)

preorder : [1, 2, 4, 5, 3, 6]
inorder  : [4, 2, 5, 1, 3, 6]
postorder: [4, 5, 2, 6, 3, 1]


### Duyệt không đệ quy bằng ngăn xếp

Đệ quy thực chất dùng *call stack*; ta có thể tự quản lý ngăn xếp. Preorder: lấy nút ra, ghi lại, đẩy con **phải trước** rồi con **trái** (để trái được lấy ra trước). Inorder: đi trái hết mức, lấy ra, rồi sang phải.

In [4]:
def preorder_lap(root):
    out, st = [], [root] if root else []
    while st:
        node = st.pop()
        out.append(node.val)
        if node.right: st.append(node.right)
        if node.left:  st.append(node.left)
    return out

def inorder_lap(root):
    out, st, cur = [], [], root
    while cur or st:
        while cur:              # đi trái hết mức
            st.append(cur); cur = cur.left
        cur = st.pop()
        out.append(cur.val)
        cur = cur.right
    return out

print("preorder lặp:", preorder_lap(cay), "| inorder lặp:", inorder_lap(cay))

preorder lặp: [1, 2, 4, 5, 3, 6] | inorder lặp: [4, 2, 5, 1, 3, 6]


### Tư duy đệ quy trên cây

Hầu hết bài về cây giải bằng cách hỏi: "nếu đã biết đáp án cho **cây con trái** và **cây con phải**, ghép lại thế nào?". Ví dụ chiều cao, số nút, tổng giá trị:

In [5]:
def chieu_cao(node):
    if node is None:
        return 0                                   # cây rỗng cao 0
    return 1 + max(chieu_cao(node.left), chieu_cao(node.right))

def so_nut(node):
    return 0 if node is None else 1 + so_nut(node.left) + so_nut(node.right)

print("chiều cao:", chieu_cao(cay), "| số nút:", so_nut(cay))

chiều cao: 3 | số nút: 6


### Lỗi hay gặp

- Quên trường hợp `node is None` → `AttributeError: 'NoneType' object has no attribute 'left'`.
- Nhầm "lá" (không có con nào) với "nút thiếu một con".
- Nhầm thứ tự preorder/inorder/postorder.
- Cây lệch (như danh sách) cao tới `n` → đệ quy sâu `n` tầng.

## 2.2 Duyệt theo chiều sâu (DFS — Depth-First Search)

### DFS là gì?

DFS đi **sâu nhất có thể** theo một nhánh, tới khi hết đường thì **quay lui** về ngã rẽ gần nhất và thử nhánh khác — giống đi mê cung bằng cách luôn bám một bức tường.

```text
        A
       / \          DFS từ A (thử hàng xóm theo thứ tự):
      B   C         A → B → D (cụt, quay lui) → E → F → C
     / \   \
    D   E - F
```

Công cụ: **đệ quy** (dùng call stack) hoặc **ngăn xếp** tự quản lý.

### Khung DFS

```text
def dfs(u):
    visited.add(u)
    xử lý u
    for v in hàng_xóm(u):
        if v not in visited:
            dfs(v)
```

In [6]:
do_thi_dfs = {
    "A": ["B", "C"], "B": ["A", "D", "E"], "C": ["A", "F"],
    "D": ["B"], "E": ["B", "F"], "F": ["C", "E"],
}

def dfs_de_quy(graph, u, visited, thu_tu, sau=0):
    visited.add(u)
    thu_tu.append(u)
    print("    " * sau + f"vào {u}")
    for v in graph[u]:
        if v not in visited:
            dfs_de_quy(graph, v, visited, thu_tu, sau + 1)
    print("    " * sau + f"rời {u}")

tt = []
dfs_de_quy(do_thi_dfs, "A", set(), tt)
print("Thứ tự DFS:", tt)

vào A
    vào B
        vào D
        rời D
        vào E
            vào F
                vào C
                rời C
            rời F
        rời E
    rời B
rời A
Thứ tự DFS: ['A', 'B', 'D', 'E', 'F', 'C']


### DFS bằng ngăn xếp (không đệ quy)

Thay hàng đợi của BFS bằng **ngăn xếp**. Tránh được giới hạn độ sâu đệ quy khi dữ liệu lớn.

In [7]:
def dfs_ngan_xep(graph, start):
    st, visited, thu_tu = [start], set(), []
    while st:
        u = st.pop()
        if u in visited:
            continue
        visited.add(u)
        thu_tu.append(u)
        for v in reversed(graph[u]):        # đảo để hàng xóm đầu tiên được lấy ra trước
            if v not in visited:
                st.append(v)
    return thu_tu

print("DFS bằng ngăn xếp:", dfs_ngan_xep(do_thi_dfs, "A"))

DFS bằng ngăn xếp: ['A', 'B', 'D', 'E', 'F', 'C']


### DFS trên lưới: tô loang (flood fill) và đếm vùng

Lưới là đồ thị ẩn: mỗi ô nối với 4 ô kề. DFS từ một ô sẽ "loang" khắp vùng liên thông chứa nó. Đếm số lần phải bắt đầu DFS mới = số vùng.

In [8]:
def dem_dao(grid):
    g = [list(row) for row in grid]
    R, C = len(g), len(g[0])
    def loang(r, c):
        if r < 0 or r >= R or c < 0 or c >= C or g[r][c] != "1":
            return                              # ra ngoài / nước / đã thăm
        g[r][c] = "#"                           # đánh dấu đã thăm
        for dr, dc in [(1, 0), (-1, 0), (0, 1), (0, -1)]:
            loang(r + dr, c + dc)
    dem = 0
    for r in range(R):
        for c in range(C):
            if g[r][c] == "1":
                dem += 1
                loang(r, c)
    return dem, ["".join(row) for row in g]

print(dem_dao(["11000",
               "11010",
               "00011"]))

(2, ['##000', '##0#0', '000##'])


### DFS trên cây: mang thông tin dọc đường đi

Trên cây không cần `visited` (không có chu trình). DFS hay **truyền xuống** thông tin của đường đi từ gốc (tổng, đường đi hiện tại) và **trả lên** kết quả của cây con.

In [9]:
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

cay_dfs = TreeNode(5, TreeNode(4, TreeNode(11, TreeNode(7), TreeNode(2))), TreeNode(8, TreeNode(13), TreeNode(4)))

def moi_duong_goc_la(node, duong, out):
    if node is None:
        return
    duong.append(node.val)                     # đi xuống: thêm nút
    if node.left is None and node.right is None:
        out.append((list(duong), sum(duong)))  # tới lá: ghi lại một bản sao
    moi_duong_goc_la(node.left, duong, out)
    moi_duong_goc_la(node.right, duong, out)
    duong.pop()                                # quay lên: bỏ nút

ket_qua = []
moi_duong_goc_la(cay_dfs, [], ket_qua)
for d, s in ket_qua:
    print(d, "tổng =", s)

[5, 4, 11, 7] tổng = 27
[5, 4, 11, 2] tổng = 22
[5, 8, 13] tổng = 26
[5, 8, 4] tổng = 17


### So sánh BFS và DFS

| | BFS | DFS |
|---|---|---|
| Cấu trúc | hàng đợi | ngăn xếp / đệ quy |
| Thứ tự | theo lớp (gần → xa) | theo nhánh (sâu trước) |
| Đường ngắn nhất (không trọng số) | **có** | không đảm bảo |
| Bộ nhớ | độ rộng lớn nhất | độ sâu lớn nhất |
| Hợp với | khoảng cách, theo tầng | liệt kê đường đi, vùng liên thông, quay lui |

Lỗi hay gặp: quên `visited` trên đồ thị có chu trình (lặp vô hạn); đệ quy quá sâu trên lưới lớn; ghi `out.append(duong)` thay vì bản sao `list(duong)` (về sau list bị sửa).

## 2.3 Quay lui (backtracking)

### Quay lui là gì?

Quay lui là cách **liệt kê / tìm kiếm có hệ thống** mọi lời giải bằng cách xây lời giải **từng bước một**. Ở mỗi bước ta thử một lựa chọn, đi tiếp; nếu đi hết (hoặc thấy không thể thành lời giải) thì **hoàn tác** lựa chọn đó và thử lựa chọn khác.

Hình dung bằng **cây quyết định**: mỗi nút là một trạng thái "đã chọn được một phần", mỗi cạnh là một lựa chọn, mỗi lá là một lời giải hoàn chỉnh. Quay lui = DFS trên cây này.

```text
Liệt kê mọi dãy độ dài 2 từ {a, b}:
                 []
            /          \
         [a]            [b]
        /   \          /   \
    [a,a] [a,b]    [b,a] [b,b]      ← 4 lá = 4 lời giải
```

### Khung chuẩn: chọn → đi tiếp → bỏ chọn

```text
def backtrack(trạng_thái):
    if trạng_thái là lời giải hoàn chỉnh:
        ghi lại BẢN SAO
        return
    for lựa_chọn in các lựa chọn hợp lệ:
        chọn          (thêm vào trạng thái)
        backtrack(trạng_thái)
        bỏ chọn       (hoàn tác — trả trạng thái về như cũ)
```

In [10]:
def moi_day(chu, k):
    out, path = [], []
    def bt(sau=0):
        if len(path) == k:
            out.append("".join(path))           # ghi bản sao
            return
        for ch in chu:
            path.append(ch)                     # chọn
            bt(sau + 1)                         # đi tiếp
            path.pop()                          # bỏ chọn
    bt()
    return out

print(moi_day("ab", 2))
print("số dãy độ dài 3 từ 'abc':", len(moi_day("abc", 3)), "= 3^3")

['aa', 'ab', 'ba', 'bb']
số dãy độ dài 3 từ 'abc': 27 = 3^3


### Ba dạng kinh điển

| Dạng | Cách chọn ở mỗi bước | Kích thước kết quả |
|---|---|---|
| **Tập con** (subset) | mỗi phần tử: lấy hoặc không; hoặc chọn phần tử tiếp theo từ chỉ số `start` | `2^n` |
| **Tổ hợp** chập `k` | chọn phần tử từ `start` trở đi (không quay lại phía trước) | `C(n, k)` |
| **Hoán vị** | chọn phần tử **chưa dùng** (mảng `used`) | `n!` |

Tham số `start` đảm bảo không sinh lại cùng một tập theo thứ tự khác (`{1,2}` và `{2,1}` là một).

In [11]:
def tap_con(nums):
    out, path = [], []
    def bt(start):
        out.append(path[:])                     # mọi nút của cây đều là một tập con
        for i in range(start, len(nums)):
            path.append(nums[i])
            bt(i + 1)
            path.pop()
    bt(0)
    return out

def hoan_vi(nums):
    out, path, used = [], [], [False] * len(nums)
    def bt():
        if len(path) == len(nums):
            out.append(path[:]); return
        for i in range(len(nums)):
            if used[i]:
                continue
            used[i] = True; path.append(nums[i])
            bt()
            path.pop(); used[i] = False
    bt()
    return out

print("tập con của [1,2,3]:", tap_con([1, 2, 3]))
print("hoán vị của [1,2,3]:", hoan_vi([1, 2, 3]))

tập con của [1,2,3]: [[], [1], [1, 2], [1, 2, 3], [1, 3], [2], [2, 3], [3]]
hoán vị của [1,2,3]: [[1, 2, 3], [1, 3, 2], [2, 1, 3], [2, 3, 1], [3, 1, 2], [3, 2, 1]]


### Vết chạy: thấy rõ "chọn / bỏ chọn"

In [12]:
def vet_tap_con(nums):
    path = []
    def bt(start, sau):
        print("    " * sau + f"path = {path}")
        for i in range(start, len(nums)):
            path.append(nums[i])
            bt(i + 1, sau + 1)
            path.pop()
    bt(0, 0)

vet_tap_con([1, 2, 3])

path = []
    path = [1]
        path = [1, 2]
            path = [1, 2, 3]
        path = [1, 3]
    path = [2]
        path = [2, 3]
    path = [3]


### Cắt tỉa (pruning) và bỏ trùng

- **Cắt tỉa**: nếu trạng thái hiện tại **chắc chắn** không dẫn tới lời giải (tổng đã vượt target, còn quá ít phần tử…), `return` sớm — bỏ cả nhánh cây.
- **Bỏ trùng** khi đầu vào có phần tử lặp: **sắp xếp** trước, rồi trong cùng một tầng (cùng vòng `for`) bỏ qua phần tử bằng phần tử ngay trước nó: `if i > start and nums[i] == nums[i-1]: continue`.

In [13]:
def tap_con_tong(nums, target):
    nums = sorted(nums)
    out, path, dem = [], [], [0]
    def bt(start, con_lai):
        dem[0] += 1
        if con_lai == 0:
            out.append(path[:]); return
        for i in range(start, len(nums)):
            if i > start and nums[i] == nums[i - 1]:
                continue                        # bỏ trùng cùng tầng
            if nums[i] > con_lai:
                break                           # cắt tỉa: đã sắp xếp nên các số sau còn lớn hơn
            path.append(nums[i])
            bt(i + 1, con_lai - nums[i])
            path.pop()
    bt(0, target)
    return out, dem[0]

print("các tập có tổng 8 từ [10,1,2,7,6,1,5]:", tap_con_tong([10, 1, 2, 7, 6, 1, 5], 8))

các tập có tổng 8 từ [10,1,2,7,6,1,5]:

 ([[1, 1, 6], [1, 2, 5], [1, 7], [2, 6]], 17)


### Lỗi hay gặp

- `out.append(path)` thay vì `out.append(path[:])` → mọi phần tử của `out` là **cùng một** list, cuối cùng đều rỗng.
- Quên **bỏ chọn** (`pop`, `used[i] = False`) → trạng thái "dính" sang nhánh khác.
- Dùng `start = i` thay vì `i + 1` (hoặc ngược lại) → lấy lặp phần tử ngoài ý muốn / thiếu lời giải.
- Bỏ trùng bằng `i > 0` thay vì `i > start` → bỏ mất lời giải hợp lệ dùng hai phần tử bằng nhau ở hai tầng khác nhau.

In [14]:
def sai_khong_sao_chep(nums):
    out, path = [], []
    def bt(start):
        out.append(path)                        # SAI: thêm chính list path
        for i in range(start, len(nums)):
            path.append(nums[i]); bt(i + 1); path.pop()
    bt(0)
    return out
print("quên sao chép:", sai_khong_sao_chep([1, 2]))

quên sao chép:

 [[], [], [], []]


## 2.4 Áp dụng vào bài này

Liệt kê mọi đường **gốc → lá** có tổng bằng target: **DFS** trên cây, mang theo **đường đi hiện tại** (thêm nút khi đi xuống, bỏ khi quay lên — đúng mẫu **quay lui**), và chỉ ghi lại **bản sao** của đường đi khi tới **lá**.

# 3. Nền tảng

In [15]:
from typing import List, Optional
from collections import deque
import random, sys
sys.setrecursionlimit(20000)


class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right


def build_tree(values: list) -> Optional[TreeNode]:
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    q = deque([root])
    i = 1
    while q and i < len(values):
        node = q.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i]); q.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i]); q.append(node.right)
        i += 1
    return root

EX = [5, 4, 8, 11, None, 13, 4, 7, 2, None, None, 5, 1]

# 4. Baseline: liệt kê mọi đường gốc → lá rồi lọc

Tạo list mới ở mỗi bước (`path + [node.val]`) — dễ hiểu, nhưng mỗi nút sao chép cả đường đi: `O(n · h)` bộ nhớ tạm.

In [16]:
def all_root_to_leaf(root):
    if not root:
        return []
    if not root.left and not root.right:
        return [[root.val]]
    return [[root.val] + p for child in (root.left, root.right) for p in all_root_to_leaf(child)]

def path_sum_filter(root, target):
    return [p for p in all_root_to_leaf(root) if sum(p) == target]

print(all_root_to_leaf(build_tree(EX)))
print(path_sum_filter(build_tree(EX), 22))

[[5, 4, 11, 7], [5, 4, 11, 2], [5, 8, 13], [5, 8, 4, 5], [5, 8, 4, 1]]
[[5, 4, 11, 2], [5, 8, 4, 5]]


# 5. Ý tưởng chính: DFS + backtracking

Một list `path` **dùng chung**:

```text
dfs(node, remain):
    if node is None: return
    path.append(node.val)
    remain -= node.val
    if node là lá and remain == 0:
        result.append(path[:])        # SAO CHÉP
    else:
        dfs(node.left, remain)
        dfs(node.right, remain)
    path.pop()                        # quay lui
```

`remain` là số nguyên (bất biến) nên mỗi lời gọi có bản riêng — không cần "hoàn tác". `path` là list dùng chung nên **phải** `pop`.

In [17]:
def path_sum_dfs(root: Optional[TreeNode], target: int) -> List[List[int]]:
    result, path = [], []
    def dfs(node, remain):
        if not node:
            return
        path.append(node.val)
        remain -= node.val
        if not node.left and not node.right and remain == 0:
            result.append(path[:])
        else:
            dfs(node.left, remain)
            dfs(node.right, remain)
        path.pop()
    dfs(root, target)
    return result

print(path_sum_dfs(build_tree(EX), 22))

[[5, 4, 11, 2], [5, 8, 4, 5]]


# 6. Mô phỏng

In [18]:
def trace_path_sum(values, target):
    root = build_tree(values)
    result, path = [], []
    def dfs(node, remain, depth):
        if not node:
            return
        path.append(node.val)
        remain -= node.val
        ind = "  " * depth
        if not node.left and not node.right:
            ok = remain == 0
            print(f"{ind}lá {node.val}: path = {path}, còn lại {remain} -> {'LƯU' if ok else 'bỏ'}")
            if ok:
                result.append(path[:])
        else:
            print(f"{ind}vào {node.val}: path = {path}, còn lại {remain}")
            dfs(node.left, remain, depth + 1)
            dfs(node.right, remain, depth + 1)
        path.pop()
    dfs(root, target, 0)
    print("Kết quả:", result)
    return result

_ = trace_path_sum(EX, 22)

vào 5: path = [5], còn lại 17
  vào 4: path = [5, 4], còn lại 13
    vào 11: path = [5, 4, 11], còn lại 2
      lá 7: path = [5, 4, 11, 7], còn lại -5 -> bỏ
      lá 2: path = [5, 4, 11, 2], còn lại 0 -> LƯU
  vào 8: path = [5, 8], còn lại 9
    lá 13: path = [5, 8, 13], còn lại -4 -> bỏ
    vào 4: path = [5, 8, 4], còn lại 5
      lá 5: path = [5, 8, 4, 5], còn lại 0 -> LƯU
      lá 1: path = [5, 8, 4, 1], còn lại 4 -> bỏ
Kết quả: [[5, 4, 11, 2], [5, 8, 4, 5]]


# 7. Cách 2: Lặp bằng stack

Mỗi phần tử stack là `(node, remain, path_tới_node)`. Sao chép đường đi ở mỗi bước — đơn giản, tránh đệ quy sâu (cây lệch 5000 nút).

In [19]:
def path_sum_iter(root: Optional[TreeNode], target: int) -> List[List[int]]:
    if not root:
        return []
    result = []
    stack = [(root, target - root.val, [root.val])]
    while stack:
        node, remain, path = stack.pop()
        if not node.left and not node.right and remain == 0:
            result.append(path)
        for child in (node.right, node.left):           # đẩy phải trước để trái ra trước
            if child:
                stack.append((child, remain - child.val, path + [child.val]))
    return result

print(path_sum_iter(build_tree(EX), 22))

[[5, 4, 11, 2], [5, 8, 4, 5]]

# 8. Phiên bản đúng format LeetCode

In [20]:
class Solution:
    def pathSum(self, root: Optional[TreeNode], targetSum: int) -> List[List[int]]:
        result, path = [], []

        def dfs(node: Optional[TreeNode], remain: int) -> None:
            if not node:
                return
            path.append(node.val)
            remain -= node.val
            if not node.left and not node.right and remain == 0:
                result.append(path[:])
            else:
                dfs(node.left, remain)
                dfs(node.right, remain)
            path.pop()

        dfs(root, targetSum)
        return result

In [21]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<20} PASS  {actual}")


def random_tree(n, lo=-3, hi=3):
    if n == 0:
        return None
    k = random.randint(0, n - 1)
    return TreeNode(random.randint(lo, hi), random_tree(k, lo, hi), random_tree(n - 1 - k, lo, hi))


sol = Solution()
check("Ví dụ 1", sol.pathSum(build_tree(EX), 22), [[5, 4, 11, 2], [5, 8, 4, 5]])
check("Ví dụ 2", sol.pathSum(build_tree([1, 2, 3]), 5), [])
check("Ví dụ 3", sol.pathSum(build_tree([1, 2]), 0), [])
check("Rỗng", sol.pathSum(None, 0), [])
check("Giá trị âm", sol.pathSum(build_tree([-2, None, -3]), -5), [[-2, -3]])
check("Gốc là lá", sol.pathSum(build_tree([7]), 7), [[7]])

random.seed(113)
for _ in range(3000):
    root = random_tree(random.randint(0, 14))
    t = random.randint(-6, 6)
    expected = sorted(path_sum_filter(root, t))
    assert sorted(sol.pathSum(root, t)) == expected == sorted(path_sum_iter(root, t))
print("3000 cây ngẫu nhiên: khớp cách liệt kê-rồi-lọc")

Ví dụ 1              PASS  [[5, 4, 11, 2], [5, 8, 4, 5]]
Ví dụ 2              PASS  []
Ví dụ 3              PASS  []
Rỗng                 PASS  []
Giá trị âm           PASS  [[-2, -3]]
Gốc là lá            PASS  [[7]]


3000 cây ngẫu nhiên: khớp cách liệt kê-rồi-lọc

# 9. Vì sao đúng? Độ phức tạp?

**Bất biến:** khi đang ở `dfs(node, remain)` (sau khi `append`), `path` là đúng các giá trị trên đường từ gốc tới `node`, và `remain = targetSum − sum(path)`. `pop` khi rời nút khôi phục bất biến cho nút cha.

Mỗi đường gốc → lá tương ứng đúng một lá, nên mỗi đường hợp lệ được ghi đúng một lần.

```text
Time:  O(n · h) xấu nhất — duyệt O(n) nút, nhưng mỗi lần lưu tốn O(h) để sao chép path
       (số đường lưu tối đa là số lá; cây đầy đủ: O(n/2 · log n))
Space: O(h) cho path và đệ quy (không tính output)
```

# 10. Những lỗi dễ mắc

## Lỗi 1: Dừng khi `remain == 0` mà chưa tới lá

In [22]:
def wrong_not_leaf(root, target):
    result, path = [], []
    def dfs(node, remain):
        if not node:
            return
        path.append(node.val)
        remain -= node.val
        if remain == 0:                              # SAI: thiếu kiểm tra lá
            result.append(path[:])
        dfs(node.left, remain)
        dfs(node.right, remain)
        path.pop()
    dfs(root, target)
    return result

print("Sai :", wrong_not_leaf(build_tree([1, 2]), 1), "| Đúng:", sol.pathSum(build_tree([1, 2]), 1))
print("Sai :", wrong_not_leaf(build_tree([1, -1, None, 1]), 0), "| Đúng:", sol.pathSum(build_tree([1, -1, None, 1]), 0))

Sai :

 [[1]] | Đúng: []
Sai : [[1, -1]] | Đúng: []


Nút `1` có con `2` nên **không** phải lá — đường `[1]` không hợp lệ. Vì giá trị có thể âm, ta cũng **không** được dừng sớm khi `remain < 0` (đi tiếp có thể quay về 0).

## Lỗi 2: Kiểm tra "lá" bằng `node is None`

In [23]:
def wrong_none_as_leaf(root, target):
    result, path = [], []
    def dfs(node, remain):
        if not node:
            if remain == 0:                          # SAI: None không phải lá
                result.append(path[:])
            return
        path.append(node.val)
        dfs(node.left, remain - node.val)
        dfs(node.right, remain - node.val)
        path.pop()
    dfs(root, target)
    return result

print("Sai :", wrong_none_as_leaf(build_tree(EX), 22))

Sai : [[5, 4, 11, 2], [5, 4, 11, 2], [5, 8, 4, 5], [5, 8, 4, 5]]


Mỗi lá có **hai** con `None` → mỗi đường được lưu **hai lần**; và nút một con (như `1` trong `[1, 2]`) cũng bị coi là kết thúc đường qua con `None` của nó.

## Lỗi 3: Lưu `path` thay vì `path[:]`

Mọi phần tử của `result` là **cùng một** list — cuối cùng đều rỗng.

In [24]:
def wrong_no_copy(root, target):
    result, path = [], []
    def dfs(node, remain):
        if not node:
            return
        path.append(node.val)
        remain -= node.val
        if not node.left and not node.right and remain == 0:
            result.append(path)                      # SAI
        else:
            dfs(node.left, remain)
            dfs(node.right, remain)
        path.pop()
    dfs(root, target)
    return result

print("Sai :", wrong_no_copy(build_tree(EX), 22))

Sai : [[], []]


## Lỗi 4: Quên `pop` khi ra khỏi nút

`path` cứ dài ra, chứa cả những nút của nhánh đã đi qua.

# 11. Mẹo nhận diện pattern

**"Mọi đường từ gốc tới lá thỏa điều kiện"** → DFS + backtracking với `path` dùng chung.

| Bài | Yêu cầu | Kỹ thuật |
|---|---|---|
| #112 Path Sum | có hay không | DFS trả bool |
| #113 Path Sum II | liệt kê | DFS + backtracking |
| #437 Path Sum III | đường bất kỳ đi xuống (không cần từ gốc / tới lá) | prefix sum + hash map |
| #257 Binary Tree Paths | in mọi đường | DFS + backtracking |
| #129 Sum Root to Leaf Numbers | ghép chữ số | DFS mang theo giá trị |

# 12. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Với cây ví dụ, liệt kê mọi đường gốc → lá và tổng của chúng. Có bao nhiêu đường có tổng `26`?

## Bài 2 — Điền chỗ trống

In [25]:
def exercise_2_path_sum(root: Optional[TreeNode], target: int) -> List[List[int]]:
    result, path = [], []
    def dfs(node, remain):
        if not node:
            return
        # TODO: append; trừ remain; nếu là lá và remain == 0 thì lưu bản sao; ngược lại đệ quy; pop
        pass
    dfs(root, target)
    return result


# assert exercise_2_path_sum(build_tree([5, 4, 8, 11, None, 13, 4, 7, 2, None, None, 5, 1]), 22) == [[5, 4, 11, 2], [5, 8, 4, 5]]
# assert exercise_2_path_sum(build_tree([1, 2]), 1) == []

## Bài 3 — Biến thể: Path Sum (LeetCode #112)

Chỉ trả `True/False`. Dừng ngay khi tìm thấy.

In [26]:
def exercise_3_has_path_sum(root: Optional[TreeNode], target: int) -> bool:
    # TODO
    pass


# assert exercise_3_has_path_sum(build_tree([5, 4, 8, 11, None, 13, 4, 7, 2, None, None, None, 1]), 22) is True
# assert exercise_3_has_path_sum(build_tree([1, 2, 3]), 5) is False
# assert exercise_3_has_path_sum(None, 0) is False

## Bài 4 — Biến thể: Binary Tree Paths (LeetCode #257)

Trả mọi đường gốc → lá dưới dạng chuỗi `"a->b->c"`.

```python
binary_tree_paths([1, 2, 3, None, 5]) -> ["1->2->5", "1->3"]
```

In [27]:
def exercise_4_binary_tree_paths(root: Optional[TreeNode]) -> List[str]:
    # TODO
    pass


# assert exercise_4_binary_tree_paths(build_tree([1, 2, 3, None, 5])) == ["1->2->5", "1->3"]
# assert exercise_4_binary_tree_paths(build_tree([1])) == ["1"]

## Bài 5 — Mini challenge: Path Sum III (LeetCode #437)

Đếm số đường đi **xuống** (từ cha tới con, không nhất thiết bắt đầu ở gốc hay kết thúc ở lá) có tổng bằng `target`. `O(n)` bằng **prefix sum + hash map**.

```python
path_sum_iii([10, 5, -3, 3, 2, None, 11, 3, -2, None, 1], 8) -> 3
```

Gợi ý nhanh: giống #560 Subarray Sum Equals K nhưng trên đường gốc → nút hiện tại. Nhớ **giảm đếm** khi quay lui.

In [28]:
def exercise_5_path_sum_iii(root: Optional[TreeNode], target: int) -> int:
    # TODO
    pass


# assert exercise_5_path_sum_iii(build_tree([10, 5, -3, 3, 2, None, 11, 3, -2, None, 1]), 8) == 3
# assert exercise_5_path_sum_iii(build_tree([5, 4, 8, 11, None, 13, 4, 7, 2, None, None, 5, 1]), 22) == 3
# assert exercise_5_path_sum_iii(build_tree([1]), 0) == 0

# 13. Gợi ý

**Bài 1**
- Mức 1: dùng `all_root_to_leaf(build_tree(EX))` sau khi tự làm.

**Bài 3**
- Mức 1: `if not root: return False`; ở lá trả `remain == root.val`; ngược lại `or` hai con.

**Bài 4**
- Mức 1: như bài chính, ở lá lưu `"->".join(map(str, path))`.

**Bài 5**
- Mức 1: `prefix` = tổng từ gốc tới nút hiện tại; số đường kết thúc tại nút = `count[prefix − target]`.
- Mức 2: `count = {0: 1}`; khi vào nút: tăng `count[prefix]`; khi ra: giảm.

# 14. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [29]:
for p in all_root_to_leaf(build_tree(EX)):
    print(p, "tổng =", sum(p))
print("Số đường tổng 26:", len(sol.pathSum(build_tree(EX), 26)))

[5, 4, 11, 7] tổng = 27
[5, 4, 11, 2] tổng = 22
[5, 8, 13] tổng = 26
[5, 8, 4, 5] tổng = 22
[5, 8, 4, 1] tổng = 18
Số đường tổng 26: 1


## Đáp án Bài 2

In [30]:
def answer_2_path_sum(root: Optional[TreeNode], target: int) -> List[List[int]]:
    return path_sum_dfs(root, target)

assert answer_2_path_sum(build_tree([5, 4, 8, 11, None, 13, 4, 7, 2, None, None, 5, 1]), 22) == [[5, 4, 11, 2], [5, 8, 4, 5]]
assert answer_2_path_sum(build_tree([1, 2]), 1) == []
print("Bài 2 PASS (giống path_sum_dfs ở mục 5)")

Bài 2 PASS (giống path_sum_dfs ở mục 5)


## Đáp án Bài 3

In [31]:
def answer_3_has_path_sum(root: Optional[TreeNode], target: int) -> bool:
    if not root:
        return False
    if not root.left and not root.right:
        return target == root.val
    rest = target - root.val
    return answer_3_has_path_sum(root.left, rest) or answer_3_has_path_sum(root.right, rest)

assert answer_3_has_path_sum(build_tree([5, 4, 8, 11, None, 13, 4, 7, 2, None, None, None, 1]), 22) is True
assert answer_3_has_path_sum(build_tree([1, 2, 3]), 5) is False
assert answer_3_has_path_sum(None, 0) is False
print("Bài 3 PASS")

Bài 3 PASS


## Đáp án Bài 4

In [32]:
def answer_4_binary_tree_paths(root: Optional[TreeNode]) -> List[str]:
    out, path = [], []
    def dfs(node):
        if not node:
            return
        path.append(str(node.val))
        if not node.left and not node.right:
            out.append("->".join(path))
        else:
            dfs(node.left)
            dfs(node.right)
        path.pop()
    dfs(root)
    return out

assert answer_4_binary_tree_paths(build_tree([1, 2, 3, None, 5])) == ["1->2->5", "1->3"]
assert answer_4_binary_tree_paths(build_tree([1])) == ["1"]
print("Bài 4 PASS")

Bài 4 PASS


## Đáp án Bài 5

In [33]:
from collections import defaultdict

def answer_5_path_sum_iii(root: Optional[TreeNode], target: int) -> int:
    count = defaultdict(int)
    count[0] = 1
    total = 0
    def dfs(node, prefix):
        nonlocal total
        if not node:
            return
        prefix += node.val
        total += count[prefix - target]
        count[prefix] += 1
        dfs(node.left, prefix)
        dfs(node.right, prefix)
        count[prefix] -= 1                       # quay lui
    dfs(root, 0)
    return total

def brute_iii(root, target):
    def from_node(node, remain):
        if not node:
            return 0
        remain -= node.val
        return (remain == 0) + from_node(node.left, remain) + from_node(node.right, remain)
    if not root:
        return 0
    return from_node(root, target) + brute_iii(root.left, target) + brute_iii(root.right, target)

assert answer_5_path_sum_iii(build_tree([10, 5, -3, 3, 2, None, 11, 3, -2, None, 1]), 8) == 3
assert answer_5_path_sum_iii(build_tree([5, 4, 8, 11, None, 13, 4, 7, 2, None, None, 5, 1]), 22) == 3
assert answer_5_path_sum_iii(build_tree([1]), 0) == 0
for _ in range(2000):
    root = random_tree(random.randint(0, 12))
    t = random.randint(-4, 4)
    assert answer_5_path_sum_iii(root, t) == brute_iii(root, t)
print("Bài 5 PASS (khớp brute force O(n²) trên 2000 cây)")

Bài 5 PASS (khớp brute force O(n²) trên 2000 cây)


# 15. Tổng kết

> DFS mang theo `path` (dùng chung, `append`/`pop`) và `remain`. Chỉ lưu `path[:]` khi tới **lá thật** (`not left and not right`) và `remain == 0`. Không dừng sớm theo tổng vì giá trị có thể âm.

---

## Checklist trước khi nộp

- [ ] Điều kiện lá: `not node.left and not node.right`?
- [ ] Lưu **bản sao** `path[:]`?
- [ ] `path.pop()` ở cuối mỗi lời gọi?
- [ ] Không cắt tỉa theo `remain < 0` (có số âm)?